# 9. Filter Parsed Texts With ModernBERT Prefilter

This notebook applies the trained ModernBERT paragraph prefilter to the full parsed-text corpus. It reads `parsed_texts/publications_*` and writes reduced files to `filtered_texts/publications_*`, which are then used by the LLM extraction workflow in `code/text_mining`.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

import re
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path

# =========================================================
# SETTINGS
# =========================================================
# Examples:
# TARGETS = 2010
# TARGETS = [2010]
# TARGETS = [2010, 2019_2]
# TARGETS = ["publications_2010", "publications_2011"]
#TARGETS = ["2019_1", "2019_2"]# TARGETS = "all"

TARGETS = [2010,2011,2012] #adjust depending on the years to process
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"

SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.

MODEL_DIR = DATA_ROOT / "training_data" / "modernbert_db_prefilter_binary_finetuned"
INPUT_ROOT = DATA_ROOT / "parsed_texts"
OUTPUT_ROOT = DATA_ROOT / "filtered_texts"
MAX_FILES_PER_TARGET = None  # e.g. 1000 for a quick test run otherwise None
PROGRESS_EVERY = 100 #specify how frequently the progress/remaining time should get reported

THRESHOLD = 0.30 #do not change (this is the best performing threshold)
BATCH_SIZE = 16 # adjust how many paragraphs get forwarded to the model (optimal setting depends on memory constraints) 
MAX_LENGTH = 512

max_paragraph_chars = 1200

def resolve_target_subfolders(targets):
    if isinstance(targets, str):
        if targets.lower() == "all":
            subfolders = sorted(
                p.name for p in INPUT_ROOT.glob("publications_*") if p.is_dir()
            )
            if not subfolders:
                raise FileNotFoundError(f"No matching subfolders found in {INPUT_ROOT}")
            return subfolders
        targets = [targets]
    elif isinstance(targets, int):
        targets = [targets]

    resolved = []

    for target in targets:
        if isinstance(target, int):
            subfolder = f"publications_{target}"
        else:
            target = str(target).strip()
            subfolder = target if target.startswith("publications_") else f"publications_{target}"

        if subfolder not in resolved:
            resolved.append(subfolder)

    if not resolved:
        raise ValueError("No target subfolders were selected.")

    return resolved


def build_target_specs(targets):
    target_specs = []

    for subfolder in resolve_target_subfolders(targets):
        input_folder = INPUT_ROOT / subfolder
        output_folder = OUTPUT_ROOT / subfolder

        if not input_folder.exists():
            raise FileNotFoundError(f"Input folder does not exist: {input_folder}")

        output_folder.mkdir(parents=True, exist_ok=True)

        target_specs.append({
            "subfolder": subfolder,
            "input_folder": input_folder,
            "output_folder": output_folder,
            "status_csv": output_folder / "reduction_status.csv"
        })

    return target_specs


TARGET_SPECS = build_target_specs(TARGETS)

# =========================================================
# PARAGRAPH CUTTING ONLY
# =========================================================
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\ufeff", " ")
    text = text.replace("\xa0", " ")
    text = text.replace("\u00ad", "")

    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def is_probable_heading(text):
    t = text.strip()
    if not t:
        return False

    if len(t) > 120:
        return False

    if re.search(r"[.!?]$", t):
        return False

    if re.match(r"^\d+(\.\d+)*\s+[A-Z][A-Za-z].*$", t):
        return True

    words = re.findall(r"[A-Za-z][A-Za-z\-]*", t)
    if not words:
        return False

    if t.isupper() and len(words) <= 12:
        return True

    capitalized_ratio = sum(w[0].isupper() for w in words) / len(words)
    if len(words) <= 10 and capitalized_ratio >= 0.8:
        return True

    return False


def merge_wrapped_lines_into_paragraphs(text):
    lines = text.split("\n")
    paragraphs = []
    current = ""

    for raw_line in lines:
        line = raw_line.strip()

        if not line:
            if current.strip():
                paragraphs.append(current.strip())
                current = ""
            continue

        if is_probable_heading(line):
            if current.strip():
                paragraphs.append(current.strip())
                current = ""
            paragraphs.append(line)
            continue

        if not current:
            current = line
            continue

        if current.endswith("-") and re.match(r"^[a-z]", line):
            current = current[:-1] + line
        else:
            current = f"{current} {line}"

    if current.strip():
        paragraphs.append(current.strip())

    return paragraphs


def split_long_paragraph(paragraph, max_chars=None):
    if not max_chars or len(paragraph) <= max_chars:
        return [paragraph]

    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", paragraph)

    if len(sentences) <= 1:
        return [paragraph]

    chunks = []
    current = ""

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if not current:
            current = sentence
            continue

        candidate = f"{current} {sentence}"

        if len(candidate) <= max_chars:
            current = candidate
        else:
            chunks.append(current.strip())
            current = sentence

    if current.strip():
        chunks.append(current.strip())

    return chunks


def split_into_paragraphs(text):
    text = clean_text(text)
    if not text:
        return []

    paragraphs = merge_wrapped_lines_into_paragraphs(text)

    final_paragraphs = []

    for p in paragraphs:
        p = clean_text(p)
        if not p:
            continue

        final_paragraphs.extend(split_long_paragraph(p, max_paragraph_chars))

    return [p for p in final_paragraphs if p.strip()]

# =========================================================
# LOAD MODEL
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# =========================================================
# MODEL PREDICTION
# =========================================================
def predict_paragraphs(texts, batch_size=BATCH_SIZE):
    all_results = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()

        for text, prob in zip(batch_texts, probs):
            pred = int(prob >= THRESHOLD)

            all_results.append({
                "text": text,
                "prob_relevant": float(prob),
                "pred_label": pred
            })

    return pd.DataFrame(all_results)

# =========================================================
# PROCESS FILES
# =========================================================
def process_file(txt_file, output_folder):
    try:
        try:
            text = txt_file.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            text = txt_file.read_text(encoding="latin-1")

        paragraphs = split_into_paragraphs(text)

        if len(paragraphs) == 0:
            reduced_text = ""
            paragraphs_kept = 0
        else:
            df_preds = predict_paragraphs(paragraphs)
            kept_paragraphs = df_preds[df_preds["pred_label"] == 1]["text"].to_list()

            reduced_text = "\n\n".join(kept_paragraphs)
            paragraphs_kept = len(kept_paragraphs)

        output_file = output_folder / txt_file.name
        output_file.write_text(reduced_text, encoding="utf-8")

        return {
            "filename": txt_file.name,
            "success": True,
            "paragraphs_total": len(paragraphs),
            "paragraphs_kept": paragraphs_kept,
            "output_chars": len(reduced_text),
            "error": ""
        }

    except Exception as e:
        return {
            "filename": txt_file.name,
            "success": False,
            "paragraphs_total": None,
            "paragraphs_kept": None,
            "output_chars": None,
            "error": str(e)
        }


In [ ]:
import time

# =========================================================
# RUN (RESUMABLE + TIMING)
# =========================================================
print(f"Using device: {device}")
print(f"Selected subfolders: {[spec['subfolder'] for spec in TARGET_SPECS]}")

overall_start = time.time()
overall_files_found = 0
overall_files_scheduled = 0

for target_index, target in enumerate(TARGET_SPECS, start=1):
    subfolder = target["subfolder"]
    input_folder = target["input_folder"]
    output_folder = target["output_folder"]
    status_csv = target["status_csv"]

    print("\n" + "=" * 60)
    print(f"[{target_index}/{len(TARGET_SPECS)}] Processing {subfolder}")
    print(f"Input folder: {input_folder}")
    print(f"Output folder: {output_folder}")

    txt_files = sorted(input_folder.glob("*.txt"))
    print(f"Found {len(txt_files)} text files.")

    if status_csv.exists():
        df_status_existing = pd.read_csv(status_csv)
        processed_files = set(
            df_status_existing.loc[df_status_existing["success"] == True, "filename"]
        )
        status_rows = df_status_existing.to_dict("records")
        print(f"Already successfully processed: {len(processed_files)}")
    else:
        processed_files = set()
        status_rows = []
        print("No existing status file found. Starting fresh.")

    files_to_process = [
        f for f in txt_files if f.name not in processed_files
    ]

    if MAX_FILES_PER_TARGET is not None:
        files_to_process = files_to_process[:MAX_FILES_PER_TARGET]

    print(f"Remaining files to process: {len(files_to_process)}")

    overall_files_found += len(txt_files)
    overall_files_scheduled += len(files_to_process)

    if len(files_to_process) == 0:
        print("Nothing to do for this subfolder.")
        continue

    progress_every = max(1, PROGRESS_EVERY)
    print(f"Progress updates every {progress_every} files.")

    start_total = time.time()

    for n, txt_file in enumerate(files_to_process, start=1):
        start_file = time.time()

        result = process_file(txt_file, output_folder)
        status_rows.append(result)

        file_time = time.time() - start_file
        elapsed_total = time.time() - start_total
        avg_time = elapsed_total / n
        remaining = len(files_to_process) - n
        eta_seconds = remaining * avg_time

        if n % progress_every == 0 or n == len(files_to_process):
            print(
                f"[{n}/{len(files_to_process)}] Last file: {txt_file.name} | "
                f"last: {file_time:.2f}s | "
                f"avg: {avg_time:.2f}s | "
                f"elapsed: {elapsed_total/60:.1f} min | "
                f"ETA: {eta_seconds/60:.1f} min"
            )

        pd.DataFrame(status_rows).to_csv(
            status_csv,
            index=False,
            encoding="utf-8-sig"
        )

    total_time = time.time() - start_total

    print(f"Finished {subfolder}.")
    print(f"Runtime: {total_time/60:.2f} minutes")
    print(f"Avg per file: {total_time / max(len(files_to_process), 1):.2f} seconds")
    print(f"Reduced text files saved to: {output_folder}")
    print(f"Status saved to: {status_csv}")

total_runtime = time.time() - overall_start

print("\nAll selected folders done.")
print(f"Folders processed: {len(TARGET_SPECS)}")
print(f"Total text files found: {overall_files_found}")
print(f"Total files scheduled in this run: {overall_files_scheduled}")
print(f"Total runtime: {total_runtime/60:.2f} minutes")
